# Getting Started with qubox: Sessions, Baseline Experiments, and Calibration Patches

This notebook is the first onboarding tutorial for the repository.
It is written for a new user who wants to understand the real current `qubox` workflow for standard cQED work.

By the end of the notebook you will know how to:

- open a session for an existing sample and cooldown,
- understand where configuration, calibration, pulse, and result files live on disk,
- inspect saved artifacts such as `calibration.json`, `pulses.json`, `measureConfig.json`, and runtime summaries,
- run or at least prepare the most common baseline experiments,
- analyze saved experiment outputs,
- preview calibration patches before applying them,
- and verify what changed after a patch is accepted.

This notebook uses the current repository reality:

- `qubox.Session` is the top-level session entry point.
- `qubox.notebook` still hosts several notebook-facing experiment classes and helpers from the active runtime stack.
- `qubox_tools` is the analysis-side package for reusable post-processing utilities.

That split is intentional here. The notebook does not invent convenience APIs that do not exist.


## 1. Environment and Imports

The imports below are the minimum set used in the rest of the tutorial.

Important objects:

- `Session` is the top-level qubox session wrapper.
- `SampleRegistry` resolves the sample/cooldown filesystem layout.
- `ResonatorSpectroscopy`, `QubitSpectroscopy`, `PowerRabi`, `TemporalRabi`, `T1Relaxation`, and `T2Ramsey` are the concrete experiment classes we will inspect.
- `Patch` is the calibration patch container used for explicit review and apply.
- `save_config_snapshot` and `save_run_summary` are artifact helpers for writing inspection files to disk.
- `qubox_tools` is used here mainly for loading saved `.npz` outputs in a reusable way.


In [ ]:
import json
import sys
from pathlib import Path
from pprint import pprint

import matplotlib
if "IPython" not in sys.modules:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "tutorials":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import qubox_tools as qt
from qubox import Session
from qubox.notebook import (
    AnalysisResult,
    Patch,
    PowerRabi,
    ProgramBuildResult,
    QubitSpectroscopy,
    ResonatorSpectroscopy,
    RunResult,
    SampleRegistry,
    T1Relaxation,
    T2Ramsey,
    TemporalRabi,
    save_config_snapshot,
    save_run_summary,
)

CONNECT_QM = True
RUN_HARDWARE = False
APPLY_PATCHES = False

REGISTRY_BASE = REPO_ROOT
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

print(f"Repo root: {REPO_ROOT}")
print(f"CONNECT_QM={CONNECT_QM}  RUN_HARDWARE={RUN_HARDWARE}  APPLY_PATCHES={APPLY_PATCHES}")


`CONNECT_QM` and `RUN_HARDWARE` are separate on purpose.

- `CONNECT_QM=True` opens the QM session so build-time inspection works with the current runtime stack.
- `RUN_HARDWARE=False` keeps the tutorial conservative by default: when a saved artifact already exists, we load and analyze that instead of launching a fresh run.
- `APPLY_PATCHES=False` means patch cells only preview proposed changes unless you explicitly opt in.


## 2. Workflow Mental Model

Before touching code, it helps to know what the main objects represent.

- A sample stores long-lived configuration shared across cooldowns: `hardware.json`, `devices.json`, and `cqed_params.json`.
- A cooldown stores run-specific state: `calibration.json`, `pulses.json`, `measureConfig.json`, `session_runtime.json`, `data/`, and `artifacts/`.
- A session wires those files into live runtime objects: calibration store, pulse manager, QM controller, and experiment bindings.
- An experiment class typically has four stages: `build_program(...)`, `run(...)`, `analyze(...)`, and `plot(...)`.
- A patch is an explicit proposed state update. Analysis can propose a patch, but qubox does not silently commit it. You preview it first and then apply it explicitly.

In other words: the physics lives in experiment classes, the persistent lab state lives in JSON files, and patches are the bridge between the two.


In [ ]:
def latest_file(directory: Path, pattern: str) -> Path | None:
    matches = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime)
    return matches[-1] if matches else None


def load_saved_run(prefix: str, runtime_dir: Path) -> tuple[RunResult, Path, Path | None]:
    npz_path = latest_file(runtime_dir, f"{prefix}_*.npz")
    if npz_path is None:
        raise FileNotFoundError(f"No runtime artifact matching {prefix!r} was found in {runtime_dir}.")
    meta_path = npz_path.with_suffix(".meta.json")
    metadata = json.loads(meta_path.read_text(encoding="utf-8")) if meta_path.exists() else {}
    output = qt.Output.from_file(str(npz_path))
    run_result = RunResult(mode="hardware", output=output, metadata=metadata)
    return run_result, npz_path, meta_path if meta_path.exists() else None


def summarize_build(build: ProgramBuildResult) -> dict[str, object]:
    sweep_sizes = {
        key: int(len(value)) if hasattr(value, "__len__") else value
        for key, value in (build.sweep_axes or {}).items()
    }
    return {
        "experiment_name": build.experiment_name,
        "builder_function": build.builder_function,
        "n_total": build.n_total,
        "params": build.params,
        "resolved_frequencies_hz": build.resolved_frequencies,
        "sweep_sizes": sweep_sizes,
    }


def patch_from_analysis(analysis: AnalysisResult, *, reason: str) -> Patch | None:
    updates = list((analysis.metadata or {}).get("proposed_patch_ops") or [])
    if not updates:
        return None
    return Patch.from_dict({"reason": reason, "updates": updates})


def print_directory(label: str, directory: Path, *, pattern: str = "*", limit: int = 12) -> None:
    print(f"\n{label}: {directory}")
    if not directory.exists():
        print("  <missing>")
        return
    entries = sorted(directory.glob(pattern))[:limit]
    if not entries:
        print("  <empty>")
        return
    for entry in entries:
        rel = entry.relative_to(REPO_ROOT) if entry.is_absolute() and REPO_ROOT in entry.parents else entry
        print(f"  - {rel}")


## 3. Locate the Sample, Cooldown, and On-Disk State

For this tutorial we use the sample and cooldown that are already checked into the repository.
That keeps the notebook reproducible and lets us inspect real saved artifacts.

If you are bringing up a brand new device tree in your own lab, the same `SampleRegistry` class is what you would use to create the sample and the cooldown directories before opening a session.


In [ ]:
registry = SampleRegistry(REGISTRY_BASE)
print("Available samples:", registry.list_samples())
print("Available cooldowns for tutorial sample:", registry.list_cooldowns(SAMPLE_ID))

SAMPLE_PATH = registry.sample_path(SAMPLE_ID)
COOLDOWN_PATH = registry.cooldown_path(SAMPLE_ID, COOLDOWN_ID)
RESOLVED_CONFIG_PATHS = registry.resolve_config_paths(SAMPLE_ID, COOLDOWN_ID)

print(f"\nSample path:   {SAMPLE_PATH}")
print(f"Cooldown path: {COOLDOWN_PATH}")
print("\nResolved config files:")
for name, file_path in sorted(RESOLVED_CONFIG_PATHS.items()):
    print(f"  - {name}: {file_path.relative_to(REPO_ROOT)}")

print_directory("Sample-level config", SAMPLE_PATH / "config")
print_directory("Cooldown-level config", COOLDOWN_PATH / "config")
print_directory("Cooldown data directory", COOLDOWN_PATH / "data")
print_directory("Cooldown artifacts directory", COOLDOWN_PATH / "artifacts")


What just happened:

- `SampleRegistry` told us where the current sample and cooldown live.
- The resolved config file list shows the important split between sample-level and cooldown-level files.
- The most important beginner directories are now visible:
  - `config/` for persistent state,
  - `data/` for explicit output saves,
  - `artifacts/` for generated inspection files, runtime artifacts, and build-hash snapshots.


## 4. Open a Session

`Session.open(...)` is the main starting point.
Under the hood it wraps the current runtime stack and gives you access to the underlying `SessionManager` through `session.session_manager` when you need lower-level control.

We keep `load_devices=False` here because the tutorial focuses on the QM / qubox path rather than unrelated instrument control.


In [ ]:
session = Session.open(
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    registry_base=REGISTRY_BASE,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    load_devices=False,
    connect=CONNECT_QM,
)

tutorial_calibration_snapshot = session.session_manager.calibration.create_in_memory_snapshot()
ctx = session.context_snapshot()

print(f"Session type: {type(session).__name__}")
print(f"Underlying runtime type: {type(session.session_manager).__name__}")
print(f"Experiment root: {Path(session.experiment_path).relative_to(REPO_ROOT)}")
print(f"Qubit element:   {ctx.qb_el} @ {ctx.qb_fq / 1e9:.6f} GHz")
print(f"Readout element: {ctx.ro_el} @ {ctx.ro_fq / 1e9:.6f} GHz")
print(f"Storage element: {ctx.st_el} @ {ctx.st_fq / 1e9:.6f} GHz")


In [ ]:
readout_setup = session.session_manager.override_readout_operation(
    element=ctx.ro_el,
    operation="readout",
    persist_measure_config=False,
)

pprint(readout_setup)


In [ ]:
transmon_params = session.session_manager.calibration.get_cqed_params("transmon")
tutorial_qb_therm_clks = getattr(transmon_params, "qb_therm_clks", None)

if tutorial_qb_therm_clks is None:
    tutorial_qb_therm_clks = 10000
    session.session_manager.calibration.set_cqed_params("transmon", qb_therm_clks=tutorial_qb_therm_clks)
    print(
        "Tutorial note: qb_therm_clks was missing from calibration.json, "
        "so the notebook injected an in-memory default of 10000 clock cycles "
        "to make T1/T2 build examples reproducible."
    )
    print("This change is only persisted if you later save/apply it explicitly.")
else:
    print(f"qb_therm_clks already present in calibration: {tutorial_qb_therm_clks}")


What just happened:

- `Session.open(...)` resolved the sample/cooldown pair and created the live runtime context.
- `session.context_snapshot()` returned the current compatibility snapshot of the active elements and calibrated frequencies.
- `override_readout_operation(...)` selected the active measurement operation that spectroscopy and time-domain experiments use for readout.
- We also checked that the qubit thermalization wait is available. In the current stack, some experiments still expect that value to live in calibration state.


## 5. Inspect What qubox Saves and Uses

The next cells answer the beginner questions:

- Where do pulse definitions live?
- Where does calibration live?
- What does the active readout configuration look like?
- Where do generated inspection files get written?


In [ ]:
calibration_store = session.session_manager.calibration

transmon_cal = calibration_store.get_cqed_params("transmon")
resonator_cal = calibration_store.get_cqed_params("resonator")
readout_pulse_cal = calibration_store.get_pulse_calibration("readout_pulse")
ref_r180_cal = calibration_store.get_pulse_calibration("ref_r180")

print("transmon calibration entry:")
pprint(transmon_cal.model_dump(exclude_none=True) if transmon_cal is not None else None)

print("\nresonator calibration entry:")
pprint(resonator_cal.model_dump(exclude_none=True) if resonator_cal is not None else None)

print("\nreadout_pulse calibration entry:")
pprint(readout_pulse_cal.model_dump(exclude_none=True) if readout_pulse_cal is not None else None)

print("\nref_r180 calibration entry:")
pprint(ref_r180_cal.model_dump(exclude_none=True) if ref_r180_cal is not None else None)


In [ ]:
readout_pulse_info = session.session_manager.pulse_mgr.get_pulseOp_by_element_op(ctx.ro_el, "readout", strict=False)
ref_r180_info = session.session_manager.pulse_mgr.get_pulseOp_by_element_op(ctx.qb_el, "ref_r180", strict=False)

pulse_summary = {
    "readout": {
        "pulse_name": getattr(readout_pulse_info, "pulse", None),
        "length_ns": getattr(readout_pulse_info, "length", None),
        "integration_weights": getattr(readout_pulse_info, "int_weights_mapping", None),
    },
    "ref_r180": {
        "pulse_name": getattr(ref_r180_info, "pulse", None),
        "length_ns": getattr(ref_r180_info, "length", None),
        "I_waveform_name": getattr(ref_r180_info, "I_wf_name", None),
        "Q_waveform_name": getattr(ref_r180_info, "Q_wf_name", None),
    },
}

pprint(pulse_summary)


In [ ]:
config_snapshot_path = save_config_snapshot(session.session_manager, tag="tutorial_open")
print("Config snapshot saved to:", config_snapshot_path.relative_to(REPO_ROOT))

snapshot_payload = json.loads(config_snapshot_path.read_text(encoding="utf-8"))
print("\nTop-level keys in the snapshot artifact:")
pprint(sorted(snapshot_payload.keys()))

print_directory("Latest build-hash artifact directories", COOLDOWN_PATH / "artifacts")
print_directory("Runtime artifacts", COOLDOWN_PATH / "artifacts" / "runtime", pattern="*")


What just happened:

- `calibration.json` told us the current long-lived calibrated numbers that experiments resolve against.
- `pulses.json` plus the pulse manager told us which named operations point to which waveform definitions.
- `save_config_snapshot(...)` wrote a human-inspectable JSON snapshot into `artifacts/`, which is a good habit before major experiments or calibration changes.


## 6. Baseline Experiments

We now walk through the baseline experiments most new users care about.

Teaching strategy in this notebook:

- We always show the real experiment class and the parameters it uses.
- When a representative saved artifact already exists in the repository, we load it and analyze it so the notebook is reproducible.
- If you set `RUN_HARDWARE=True`, the same cells can launch fresh runs instead.

This is also where the current API split is most visible:

- `qubox.Session` is the session entry point.
- Many experiment implementations still live under `qubox.notebook`.


### 6.1 Resonator Spectroscopy

**Physical question:** where is the readout resonator resonance?

**Important parameters:**

- `readout_op`: which measurement pulse to use,
- `rf_begin`, `rf_end`, `df`: sweep range and spacing,
- `n_avg`: averaging count,
- `ro_therm_clks`: readout thermalization wait.

Fast-path template equivalent when you only want the simplest call:

```python
result = session.exp.resonator.spectroscopy(
    readout=ctx.ro_el,
    freq=session.sweep.linspace(-5e6, 5e6, 101, center="readout"),
    n_avg=200,
    readout_op="readout",
)
```

In the cells below we use the explicit experiment class because it exposes `build_program`, `run`, `analyze`, and `plot` separately.


In [ ]:
resonator_spec = ResonatorSpectroscopy(session.session_manager)
resonator_build = resonator_spec.build_program(
    readout_op="readout",
    rf_begin=8.590e9,
    rf_end=8.600e9,
    df=2.0e5,
    n_avg=1,
    ro_therm_clks=1000,
)

pprint(summarize_build(resonator_build))


In [ ]:
if RUN_HARDWARE:
    resonator_run = resonator_spec.run(
        readout_op="readout",
        rf_begin=8.590e9,
        rf_end=8.600e9,
        df=2.0e5,
        n_avg=200,
        ro_therm_clks=1000,
    )
    resonator_source = "fresh hardware run"
else:
    resonator_run, resonator_npz, resonator_meta = load_saved_run(
        "ResonatorSpectroscopy",
        COOLDOWN_PATH / "artifacts" / "runtime",
    )
    resonator_source = f"saved artifact: {resonator_npz.relative_to(REPO_ROOT)}"

resonator_analysis = resonator_spec.analyze(resonator_run, update_calibration=True)
resonator_summary = save_run_summary(session.session_manager, resonator_run, tag="tutorial_resonator_spec")

print(resonator_source)
print("Output keys:", list(resonator_run.output.keys()))
print("Summary saved to:", resonator_summary.relative_to(REPO_ROOT))
print("Fit metrics:")
pprint(resonator_analysis.metrics)
print("\nProposed patch ops:")
pprint(resonator_analysis.metadata.get("proposed_patch_ops", []))

resonator_spec.plot(resonator_analysis)


What just happened:

- `build_program(...)` produced a `ProgramBuildResult`, which is the exact compiled experiment specification qubox is about to execute.
- `RunResult` held the actual output arrays (`I`, `Q`, `S`, `magnitude`, `frequencies`, depending on the experiment).
- `analyze(...)` fit the resonance and proposed a patch for the resonator frequency, but did not apply it yet.


### 6.2 Qubit Spectroscopy

**Physical question:** where is the qubit transition frequency?

**Important parameters:**

- `pulse`: the drive pulse or operation used during the scan,
- `qb_gain`, `qb_len`: drive amplitude and duration,
- `rf_begin`, `rf_end`, `df`: frequency sweep,
- `qb_therm_clks`: qubit thermalization wait.

Fast-path template equivalent:

```python
result = session.exp.qubit.spectroscopy(
    qubit=ctx.qb_el,
    readout=ctx.ro_el,
    freq=session.sweep.linspace(-5e6, 5e6, 101, center="qubit.ge"),
    drive_amp=0.01,
    pulse="const",
    qb_len=5000,
    qb_therm_clks=tutorial_qb_therm_clks,
    n_avg=200,
)
```


In [ ]:
qubit_spec = QubitSpectroscopy(session.session_manager)
qubit_build = qubit_spec.build_program(
    pulse="const",
    rf_begin=6.145e9,
    rf_end=6.155e9,
    df=2.0e5,
    qb_gain=0.01,
    qb_len=5000,
    n_avg=1,
    qb_therm_clks=tutorial_qb_therm_clks,
)

pprint(summarize_build(qubit_build))


In [ ]:
if RUN_HARDWARE:
    qubit_run = qubit_spec.run(
        pulse="const",
        rf_begin=6.145e9,
        rf_end=6.155e9,
        df=2.0e5,
        qb_gain=0.01,
        qb_len=5000,
        n_avg=200,
        qb_therm_clks=tutorial_qb_therm_clks,
    )
    qubit_source = "fresh hardware run"
else:
    qubit_run, qubit_npz, qubit_meta = load_saved_run(
        "qubitSpectroscopy",
        COOLDOWN_PATH / "artifacts" / "runtime",
    )
    qubit_source = f"saved artifact: {qubit_npz.relative_to(REPO_ROOT)}"

qubit_analysis = qubit_spec.analyze(qubit_run, update_calibration=True)
qubit_summary = save_run_summary(session.session_manager, qubit_run, tag="tutorial_qubit_spec")

print(qubit_source)
print("Output keys:", list(qubit_run.output.keys()))
print("Summary saved to:", qubit_summary.relative_to(REPO_ROOT))
print("Fit metrics:")
pprint(qubit_analysis.metrics)
print("\nProposed patch ops:")
pprint(qubit_analysis.metadata.get("proposed_patch_ops", []))

qubit_spec.plot(qubit_analysis)


Qubit spectroscopy is often the first place where users notice the distinction between:

- raw outputs such as complex `S`, and
- analysis metrics such as the fitted `f0` and linewidth.

The raw arrays are what came back from the run. The fitted metrics are the interpretation layer on top.


### 6.3 Power Rabi

**Physical question:** what drive amplitude corresponds to a pi pulse?

**Important parameters:**

- `op`: which calibrated pulse family to scale,
- `max_gain`, `dg`: the amplitude sweep,
- `n_avg` and `qb_therm_clks`.

This is a canonical example of an experiment that naturally proposes a calibration patch.
Its analysis usually proposes an update to the pulse amplitude and a trigger to rebuild the pulse-dependent configuration.


In [ ]:
power_rabi = PowerRabi(session.session_manager)
power_rabi_build = power_rabi.build_program(
    max_gain=1.2,
    dg=0.02,
    op="ref_r180",
    n_avg=1,
    qb_therm_clks=tutorial_qb_therm_clks,
)

pprint(summarize_build(power_rabi_build))


In [ ]:
if RUN_HARDWARE:
    power_rabi_run = power_rabi.run(
        max_gain=1.2,
        dg=0.02,
        op="ref_r180",
        n_avg=200,
        qb_therm_clks=tutorial_qb_therm_clks,
    )
    power_rabi_source = "fresh hardware run"
else:
    power_rabi_run, power_rabi_npz, power_rabi_meta = load_saved_run(
        "powerRabi",
        COOLDOWN_PATH / "artifacts" / "runtime",
    )
    power_rabi_source = f"saved artifact: {power_rabi_npz.relative_to(REPO_ROOT)}"

power_rabi_analysis = power_rabi.analyze(power_rabi_run, update_calibration=True)
power_rabi_summary = save_run_summary(session.session_manager, power_rabi_run, tag="tutorial_power_rabi")

print(power_rabi_source)
print("Output keys:", list(power_rabi_run.output.keys()))
print("Summary saved to:", power_rabi_summary.relative_to(REPO_ROOT))
print("Fit metrics:")
pprint(power_rabi_analysis.metrics)
print("\nProposed patch ops:")
pprint(power_rabi_analysis.metadata.get("proposed_patch_ops", []))

power_rabi.plot(power_rabi_analysis)


Power Rabi is usually the easiest place to see qubox's calibration philosophy:

- run the experiment,
- fit the result,
- produce a patch proposal,
- review it,
- and apply it explicitly only when you accept it.


### 6.4 Temporal Rabi

**Physical question:** what pulse duration gives the desired rotation?

The API is similar to Power Rabi, but the swept axis is pulse length instead of amplitude.

This repository does not currently ship a saved Temporal Rabi artifact for the tutorial sample, so the cell below always shows the build path and only performs execution when `RUN_HARDWARE=True`.


In [ ]:
temporal_rabi = TemporalRabi(session.session_manager)
temporal_rabi_build = temporal_rabi.build_program(
    pulse="const",
    pulse_len_begin=16,
    pulse_len_end=256,
    dt=4,
    pulse_gain=1.0,
    n_avg=1,
    qb_therm_clks=tutorial_qb_therm_clks,
)

pprint(summarize_build(temporal_rabi_build))

if RUN_HARDWARE:
    temporal_rabi_run = temporal_rabi.run(
        pulse="const",
        pulse_len_begin=16,
        pulse_len_end=256,
        dt=4,
        pulse_gain=1.0,
        n_avg=200,
        qb_therm_clks=tutorial_qb_therm_clks,
    )
    temporal_rabi_analysis = temporal_rabi.analyze(temporal_rabi_run)
    pprint(temporal_rabi_analysis.metrics)
    temporal_rabi.plot(temporal_rabi_analysis)
else:
    print("Set RUN_HARDWARE=True to execute Temporal Rabi on the connected QM system.")


### 6.5 T1 Relaxation

**Physical question:** how fast does the excited-state population decay back to the ground state?

The analysis stage can propose both:

- a coherence update (`T1`, `T1_us`), and
- a new qubit thermalization wait if you ask it to derive one.


In [ ]:
t1_exp = T1Relaxation(session.session_manager)
t1_build = t1_exp.build_program(
    delay_end=20_000,
    dt=500,
    n_avg=1,
)

pprint(summarize_build(t1_build))

if RUN_HARDWARE:
    t1_run = t1_exp.run(
        delay_end=20_000,
        dt=500,
        n_avg=200,
    )
    t1_analysis = t1_exp.analyze(
        t1_run,
        update_calibration=True,
        p0=[0, 10, 0],
        p0_time_unit="us",
        derive_qb_therm_clks=True,
        clock_period_ns=4.0,
    )
    pprint(t1_analysis.metrics)
    pprint(t1_analysis.metadata.get("proposed_patch_ops", []))
    t1_exp.plot(t1_analysis)
else:
    print("Set RUN_HARDWARE=True to execute T1 and generate a fresh coherence patch proposal.")


### 6.6 Ramsey (T2*)

**Physical question:** what is the qubit dephasing time, and is the drive slightly detuned from the true qubit frequency?

Ramsey is especially useful because the analysis can optionally propose a frequency correction in addition to a coherence update.


In [ ]:
ramsey_exp = T2Ramsey(session.session_manager)
ramsey_build = ramsey_exp.build_program(
    qb_detune=int(0.2e6),
    delay_end=20_000,
    dt=100,
    n_avg=1,
    qb_detune_MHz=0.2,
    qb_therm_clks=tutorial_qb_therm_clks,
)

pprint(summarize_build(ramsey_build))

if RUN_HARDWARE:
    ramsey_run = ramsey_exp.run(
        qb_detune=int(0.2e6),
        delay_end=20_000,
        dt=100,
        n_avg=200,
        qb_detune_MHz=0.2,
        qb_therm_clks=tutorial_qb_therm_clks,
    )
    ramsey_analysis = ramsey_exp.analyze(
        ramsey_run,
        update_calibration=True,
        p0=[0, 20, 1.0, 0.2, 0.0, 0],
        p0_time_unit="us",
        p0_freq_unit="MHz",
        apply_frequency_correction=True,
        freq_correction_sign=-1.0,
    )
    pprint(ramsey_analysis.metrics)
    pprint(ramsey_analysis.metadata.get("proposed_patch_ops", []))
    ramsey_exp.plot(ramsey_analysis)
else:
    print("Set RUN_HARDWARE=True to execute Ramsey and preview a detuning/frequency-correction patch.")


## 7. What the Main Objects Actually Are

A beginner usually becomes productive much faster once the core object types stop feeling abstract.


In [ ]:
print("session:", type(session))
print("underlying runtime:", type(session.session_manager))
print("resonator_build:", type(resonator_build))
print("resonator_run:", type(resonator_run))
print("resonator_analysis:", type(resonator_analysis))


Read those types as follows:

- `Session` is the user entry point.
- `SessionManager` is the current underlying runtime container.
- `ProgramBuildResult` is the compiled program plus provenance and sweep metadata.
- `RunResult` is the raw execution output.
- `AnalysisResult` is the interpreted result: fits, metrics, and optional proposed patches.


## 8. Calibration Workflow: Preview First, Apply Explicitly

The repository does not silently overwrite calibration state after every fit.
The intended beginner workflow is:

1. analyze with `update_calibration=True`,
2. inspect `proposed_patch_ops`,
3. convert them into a `Patch`,
4. preview with `dry_run=True`,
5. apply only when satisfied.

We demonstrate that flow using the Power Rabi analysis we already computed above.


In [ ]:
power_rabi_patch = patch_from_analysis(
    power_rabi_analysis,
    reason="tutorial_power_rabi_patch",
)

current_ref_r180 = session.session_manager.calibration.get_pulse_calibration("ref_r180")
print("Current ref_r180 amplitude:", getattr(current_ref_r180, "amplitude", None))

if power_rabi_patch is None:
    print("No patch was proposed by the Power Rabi analysis.")
    power_rabi_preview = None
else:
    power_rabi_preview = session.session_manager.orchestrator.apply_patch(
        power_rabi_patch,
        dry_run=True,
    )
    print("Dry-run patch preview:")
    pprint(power_rabi_preview)


In [ ]:
if power_rabi_patch is None:
    print("There is no patch to apply.")
elif APPLY_PATCHES:
    apply_result = session.session_manager.orchestrator.apply_patch(
        power_rabi_patch,
        dry_run=False,
    )
    updated_ref_r180 = session.session_manager.calibration.get_pulse_calibration("ref_r180")
    print("Patch applied:")
    pprint(apply_result)
    print("Updated ref_r180 amplitude:", getattr(updated_ref_r180, "amplitude", None))
else:
    print("APPLY_PATCHES=False, so the calibration store was left unchanged.")
    print("Set APPLY_PATCHES=True only after you review the preview above.")


The patch preview is one of the most important beginner habits in qubox.
It answers the question: what exactly will be written if I accept this calibration?


## 9. One-Call Calibration Runs with the Orchestrator

For fresh hardware runs, the `CalibrationOrchestrator` can execute the whole flow for you:

- run the experiment,
- persist the runtime artifact,
- analyze it,
- build the patch,
- and return a dry-run preview.

The cell below is hardware-gated on purpose because it launches a new run.


In [ ]:
if RUN_HARDWARE:
    t1_cycle = session.session_manager.orchestrator.run_analysis_patch_cycle(
        T1Relaxation(session.session_manager),
        run_kwargs={
            "delay_end": 20_000,
            "dt": 500,
            "n_avg": 200,
        },
        analyze_kwargs={
            "update_calibration": True,
            "p0": [0, 10, 0],
            "p0_time_unit": "us",
            "derive_qb_therm_clks": True,
            "clock_period_ns": 4.0,
        },
        apply=False,
        persist_artifact=True,
    )
    pprint(t1_cycle["dry_run"])
else:
    print(
        "When you are ready for a live calibration run, set RUN_HARDWARE=True "
        "and use orchestrator.run_analysis_patch_cycle(...) as the single-call helper."
    )


## 10. Inspect What Changed on Disk

After running experiments or writing summaries, the most useful beginner check is simply to inspect the latest files again.
That closes the loop between notebook objects and the repository state on disk.


In [ ]:
print_directory("Cooldown config directory", COOLDOWN_PATH / "config")
print_directory("Cooldown runtime artifacts", COOLDOWN_PATH / "artifacts" / "runtime", pattern="*")
print_directory("Cooldown data directory", COOLDOWN_PATH / "data", pattern="*")

latest_summary = latest_file(COOLDOWN_PATH / "artifacts", "run_summary_tutorial_*.json")
if latest_summary is None:
    latest_summary = latest_file(COOLDOWN_PATH / "artifacts", "run_summary_*.json")

print("\nMost recent tutorial config snapshot:", config_snapshot_path.relative_to(REPO_ROOT))
if latest_summary is not None:
    print("Most recent run summary:", latest_summary.relative_to(REPO_ROOT))
else:
    print("No run summary file found.")


## 11. Recommended Beginner Workflow

A practical order of operations for a new cooldown is:

1. Open the session and verify the sample/cooldown paths.
2. Inspect `calibration.json`, `pulses.json`, and `measureConfig.json`.
3. Save a config snapshot before making changes.
4. Run resonator spectroscopy to confirm the readout frequency.
5. Run qubit spectroscopy to confirm the qubit transition.
6. Run Power Rabi to update the pi-pulse amplitude.
7. Run Temporal Rabi, T1, and Ramsey as needed for pulse-length and coherence checks.
8. Review patch previews carefully.
9. Apply only the updates you actually want.
10. Re-run the critical baseline experiments after any meaningful calibration change.


## 12. Next Steps

Once this workflow feels comfortable, the next useful places to go are:

- `notebooks/post_cavity_experiment_context.ipynb` for a much broader end-to-end lab notebook,
- `notebooks/post_cavity_experiment_quantum_circuit.ipynb` for the circuit-oriented path,
- `API_REFERENCE.md` for the detailed current API surface,
- and `docs/qubox_tools_analysis_split.md` if you want to understand what belongs in `qubox_tools` versus the runtime stack.


## Optional Cleanup

The final code cell below closes the session.

It also restores the original in-memory calibration snapshot when `APPLY_PATCHES=False`, so tutorial-only helper changes such as the temporary `qb_therm_clks` injection are not persisted accidentally.


In [ ]:
if not APPLY_PATCHES:
    session.session_manager.calibration.restore_in_memory_snapshot(tutorial_calibration_snapshot)
    print("Restored the original in-memory calibration snapshot.")

    session.close()
    print("Session closed.")
else:
    session.close()
    print("Session closed with applied patches preserved.")
